In [ ]:
# 0. Environment setup and STAC client initialisation
import os

# ── PROJ conflict workaround ───────────────────────────────────────────────────
# Some installations (e.g. PostGIS / PostgreSQL 16) globally override PROJ_DATA
# with an outdated proj.db, which causes rasterio and pyproj to fail on import.
# Uncomment and set the path below only if you encounter proj.db version errors.
#
# _PROJ_PATH = r"path/to/your/conda_env/Lib/site-packages/pyproj/proj_dir/share/proj"
# os.environ["PROJ_DATA"] = _PROJ_PATH
# os.environ["PROJ_LIB"]  = _PROJ_PATH
# ──────────────────────────────────────────────────────────────────────────────

import pystac_client
import planetary_computer
import stackstac
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass

# Microsoft Planetary Computer -- public STAC catalogue, no credentials required.
# COG download URLs are auto-signed by the planetary_computer modifier.
STAC_URL   = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "sentinel-2-l2a"

# Bands required to compute all six spectral indices
# B02=Blue, B03=Green, B04=Red, B05=RedEdge1, B8A=NIR narrow, B11=SWIR1, B12=SWIR2
BANDS_NEEDED = ["B02", "B03", "B04", "B05", "B8A", "B11", "B12"]

catalog = pystac_client.Client.open(
    STAC_URL,
    modifier=planetary_computer.sign_inplace,  # auto-signs COG asset URLs
)

print("=" * 55)
print("  STAC client initialised successfully")
print("=" * 55)
print(f"  Catalogue : {STAC_URL}")
print(f"  Collection: {COLLECTION}")
print(f"  Bands     : {BANDS_NEEDED}")
print(f"  Auth      : Planetary Computer (no credentials needed)")
print("=" * 55)
print("\nSTAC client ready. Run Cell 1 to load the AOI.")


In [ ]:
# 1. AOI loading from GeoPackage
import geopandas as gpd
import folium
import json
from pathlib import Path

# ── USER CONFIGURATION ────────────────────────────────────────────────────────
GPKG_PATH  = Path("path/to/your/aoi.gpkg")  # <- only line to edit
LAYER_NAME = None                             # None = first layer in the file
# ─────────────────────────────────────────────────────────────────────────────


def load_aoi(gpkg_path: Path, layer_name: str | None = None) -> dict:
    """
    Load a GeoPackage, reproject to EPSG:4326, dissolve all features into a
    single polygon, and return geometry metadata used by all downstream cells.

    The STAC API consumes a [W, S, E, N] bounding box in EPSG:4326 (pystac_client
    native format), and rasterio_mask uses GeoJSON geometries for polygon clipping.
    Both are derived here from the dissolved AOI.
    """
    gdf = gpd.read_file(gpkg_path, layer=layer_name)

    print("=" * 50)
    print(f"Original CRS    : {gdf.crs}")
    print(f"Feature count   : {len(gdf)}")
    print(f"Columns         : {list(gdf.columns)}")
    print(f"Geometry types  : {gdf.geom_type.unique().tolist()}")
    print("=" * 50)

    if gdf.crs is None:
        raise ValueError("The GeoPackage has no CRS defined. Assign one before continuing.")

    if gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
        print("Reprojected to EPSG:4326")
    else:
        print("Already in EPSG:4326 -- no reprojection needed")

    aoi_dissolved = gdf.dissolve()
    aoi_geom      = aoi_dissolved.geometry.iloc[0]

    area_ha = (
        gdf.to_crs(epsg=3857)
        .dissolve()
        .geometry
        .area
        .iloc[0]
        / 10_000
    )

    bounds = aoi_geom.bounds  # (minx, miny, maxx, maxy)

    print(f"\nBounding box (lon/lat):")
    print(f"  West  : {bounds[0]:.6f}")
    print(f"  South : {bounds[1]:.6f}")
    print(f"  East  : {bounds[2]:.6f}")
    print(f"  North : {bounds[3]:.6f}")
    print(f"  Area  : {area_ha:.2f} ha")

    if aoi_geom.geom_type == "MultiPolygon":
        aoi_geom = max(aoi_geom.geoms, key=lambda g: g.area)
        print("\nMultiPolygon detected -- using the largest polygon")

    # pystac_client expects bbox as [West, South, East, North] (GeoJSON standard)
    bbox_stac = [bounds[0], bounds[1], bounds[2], bounds[3]]

    exterior_coords = list(aoi_geom.exterior.coords)
    wkt_coords = ", ".join(f"{x} {y}" for x, y in exterior_coords)
    wkt = f"POLYGON(({wkt_coords}))"

    return {
        "wkt"         : wkt,
        "bbox_stac"   : bbox_stac,
        "geojson"     : json.loads(aoi_dissolved.to_json()),
        "bounds"      : bounds,
        "area_ha"     : round(area_ha, 2),
        "crs_original": str(gdf.crs),
        "n_features"  : len(gdf),
        "gdf"         : gdf,
    }


def preview_aoi(aoi_info: dict) -> folium.Map:
    """Render the AOI on an interactive satellite basemap."""
    bounds     = aoi_info["bounds"]
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=13,
        tiles=(
            "https://server.arcgisonline.com/ArcGIS/rest/services"
            "/World_Imagery/MapServer/tile/{z}/{y}/{x}"
        ),
        attr="Esri World Imagery"
    )

    folium.GeoJson(
        aoi_info["geojson"],
        name="AOI",
        style_function=lambda _: {
            "fillColor"  : "#00ff88",
            "color"      : "#00cc66",
            "weight"     : 2,
            "fillOpacity": 0.2,
        }
    ).add_to(m)

    folium.Marker(
        location=[center_lat, center_lon],
        popup=folium.Popup(
            f"<b>Area:</b> {aoi_info['area_ha']} ha<br>"
            f"<b>Original CRS:</b> {aoi_info['crs_original']}<br>"
            f"<b>Features:</b> {aoi_info['n_features']}",
            max_width=250
        ),
        icon=folium.Icon(color="green", icon="leaf")
    ).add_to(m)

    folium.LayerControl().add_to(m)
    return m


# ------------------------------------------------------------
# EXECUTION
# ------------------------------------------------------------
aoi_info = load_aoi(GPKG_PATH, LAYER_NAME)

print(f"\nSTAC bbox : {aoi_info['bbox_stac']}")
print(f"WKT (first 200 chars): {aoi_info['wkt'][:200]}...")

mapa = preview_aoi(aoi_info)
display(mapa)


In [ ]:
# 2. Sentinel-2 L2A scene search -- Microsoft Planetary Computer STAC
import pystac_client
import planetary_computer
import pandas as pd
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional, List


@dataclass
class STACSearchConfig:
    """
    STAC search parameters for the Sentinel-2 L2A collection on Planetary Computer.

    Attributes
    ----------
    date_start      : Search window start date (ISO format YYYY-MM-DD).
    date_end        : Search window end date (ISO format YYYY-MM-DD).
    max_cloud_cover : Maximum scene-level cloud cover percentage.
    max_results     : Maximum number of STAC items to retrieve.
    collection      : STAC collection ID (default: sentinel-2-l2a).
    stac_url        : Planetary Computer STAC endpoint.
    """
    date_start:      str
    date_end:        str
    max_cloud_cover: float
    max_results:     int  = 100
    collection:      str  = "sentinel-2-l2a"
    stac_url:        str  = "https://planetarycomputer.microsoft.com/api/stac/v1"

    def __post_init__(self) -> None:
        fmt = "%Y-%m-%d"
        try:
            t_start = datetime.strptime(self.date_start, fmt)
            t_end   = datetime.strptime(self.date_end,   fmt)
        except ValueError as exc:
            raise ValueError(f"Invalid date format. Expected YYYY-MM-DD. Got: {exc}")

        if t_start >= t_end:
            raise ValueError(
                f"date_start ({self.date_start}) must be earlier than "
                f"date_end ({self.date_end})."
            )
        if not (0.0 <= self.max_cloud_cover <= 100.0):
            raise ValueError(
                f"max_cloud_cover must be in [0, 100]. Got: {self.max_cloud_cover}"
            )


class STACSceneSearch:
    """
    Queries the Microsoft Planetary Computer STAC API for Sentinel-2 L2A scenes.

    Compared to raw HTTP APIs, pystac_client handles pagination, date formatting,
    and property filtering automatically. Scene assets are Cloud Optimised GeoTIFFs
    (COGs) with auto-signed URLs -- no further authentication is required after the
    initial catalogue connection.
    """

    def __init__(self, config: STACSearchConfig) -> None:
        self._config  = config
        self._catalog = pystac_client.Client.open(
            config.stac_url,
            modifier=planetary_computer.sign_inplace,
        )

    def _extract_tile_id(self, item) -> str:
        """Extract the MGRS tile identifier from a STAC item's properties."""
        tile = item.properties.get("s2:mgrs_tile", "")
        if tile:
            return f"T{tile}" if not tile.startswith("T") else tile
        parts = item.id.split("_")
        for part in parts:
            if len(part) == 6 and part.startswith("T") and part[1:].isalnum():
                return part
        return "UNKNOWN"

    def _item_to_record(self, item) -> dict:
        """
        Convert a STAC Item to a flat dictionary for DataFrame construction.

        Key fields:
            scene_id    : Unique STAC item identifier.
            tile_id     : MGRS tile (e.g. T30SVF).
            date        : Acquisition date YYYY-MM-DD.
            cloud_cover : Scene-level cloud cover percentage.
            band_urls   : Dict mapping band name to signed COG URL.
            stac_item   : Full STAC Item, passed to stackstac in Cell 3.
        """
        cloud_cover = float(item.properties.get("eo:cloud_cover", -1.0))
        acq_date    = item.properties.get("datetime", "")[:10]
        tile_id     = self._extract_tile_id(item)

        band_urls = {
            band: item.assets[band].href
            for band in BANDS_NEEDED
            if band in item.assets
        }

        return {
            "scene_id"       : item.id,
            "scene_name"     : item.id,
            "tile_id"        : tile_id,
            "date"           : acq_date,
            "cloud_cover_pct": cloud_cover,
            "band_urls"      : band_urls,
            "stac_item"      : item,
            "origin"         : "STAC_PC",
        }

    def search(self, aoi_info: dict) -> pd.DataFrame:
        """
        Execute the STAC search and return a chronologically sorted DataFrame.

        Parameters
        ----------
        aoi_info : dict
            Output of ``load_aoi()`` from Cell 1. Must contain ``bbox_stac``.
        """
        if "bbox_stac" not in aoi_info:
            raise ValueError("aoi_info is missing 'bbox_stac'. Run Cell 1 first.")

        bbox       = aoi_info["bbox_stac"]
        date_range = f"{self._config.date_start}/{self._config.date_end}"

        print("Searching STAC catalogue...")
        print(f"  Collection : {self._config.collection}")
        print(f"  Bbox       : {bbox}")
        print(f"  Date range : {date_range}")
        print(f"  Max cloud  : {self._config.max_cloud_cover}%")

        search = self._catalog.search(
            collections = [self._config.collection],
            bbox        = bbox,
            datetime    = date_range,
            query       = {"eo:cloud_cover": {"lt": self._config.max_cloud_cover}},
            max_items   = self._config.max_results,
            sortby      = "+datetime",
        )

        items = list(search.items())

        if not items:
            print(
                f"No scenes found for the given AOI, date range "
                f"{self._config.date_start}-{self._config.date_end}, "
                f"and cloud cover < {self._config.max_cloud_cover}%."
            )
            return pd.DataFrame()

        records = [self._item_to_record(item) for item in items]
        df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
        return df


class STACSceneDeduplicator:
    """
    Removes duplicate scenes from the STAC search results.

    Deduplication strategy (applied in order):
        1. Same tile + date + satellite -> keep the most recently processed scene.
        2. Same tile + date, different satellite -> keep the lowest cloud cover.
    """

    def __init__(self, scenes_df: pd.DataFrame) -> None:
        if scenes_df.empty:
            raise ValueError("DataFrame is empty. Run STACSceneSearch.search() first.")
        self._df = scenes_df.copy()

    def _extract_satellite(self, scene_name: str) -> str:
        """Extract satellite identifier (S2A, S2B, S2C) from the scene name."""
        try:
            return scene_name.split("_")[0]
        except (IndexError, AttributeError):
            return "UNKNOWN"

    def _extract_processing_ts(self, scene_name: str) -> str:
        """Extract the processing timestamp (last token) from the scene name."""
        try:
            return scene_name.split("_")[-1]
        except (IndexError, AttributeError):
            return ""

    def deduplicate(self) -> pd.DataFrame:
        """Apply both deduplication passes and return the cleaned DataFrame."""
        df = self._df.copy()
        df["satellite"]     = df["scene_name"].apply(self._extract_satellite)
        df["processing_ts"] = df["scene_name"].apply(self._extract_processing_ts)

        # Pass 1 -- latest processing timestamp per tile + date + satellite
        df = (
            df.sort_values("processing_ts", ascending=False)
            .drop_duplicates(subset=["date", "tile_id", "satellite"], keep="first")
            .reset_index(drop=True)
        )

        # Pass 2 -- lowest cloud cover per tile + date (across satellites)
        df = (
            df.sort_values("cloud_cover_pct", ascending=True)
            .drop_duplicates(subset=["date", "tile_id"], keep="first")
            .reset_index(drop=True)
        )

        df = df.drop(columns=["satellite", "processing_ts"])
        return df.sort_values("date").reset_index(drop=True)


def summarise_results(df: pd.DataFrame) -> None:
    """Print a summary of the raw STAC search results."""
    if df.empty:
        print("No results found."); return

    print("=" * 60)
    print(f"Total scenes found  : {len(df)}")
    print(f"Date range          : {df['date'].min()} to {df['date'].max()}")
    print(f"Unique MGRS tiles   : {sorted(df['tile_id'].unique().tolist())}")
    print(f"Mean cloud cover    : {df['cloud_cover_pct'].mean():.1f}%")
    print(f"Min cloud cover     : {df['cloud_cover_pct'].min():.1f}%")
    print("=" * 60)

    if df["tile_id"].nunique() > 1:
        print(
            f"NOTE: AOI spans {df['tile_id'].nunique()} MGRS tiles. "
            f"Cell 3 will handle the automatic tile merge."
        )

    print("\nAvailable scenes:")
    print(df[["date", "tile_id", "cloud_cover_pct", "scene_name"]].to_string(index=True))


def summarise_deduplicated(original_df: pd.DataFrame, clean_df: pd.DataFrame) -> None:
    """Print a before/after comparison of the deduplication step."""
    removed = len(original_df) - len(clean_df)
    print("=" * 60)
    print(f"Scenes before deduplication : {len(original_df)}")
    print(f"Scenes after deduplication  : {len(clean_df)}")
    print(f"Duplicates removed          : {removed}")
    print(f"Date range                  : {clean_df['date'].min()} to {clean_df['date'].max()}")
    print(f"Mean cloud cover            : {clean_df['cloud_cover_pct'].mean():.2f}%")
    print("=" * 60)
    print("\nClean scene list:")
    print(
        clean_df[["date", "tile_id", "cloud_cover_pct", "scene_name"]]
        .to_string(index=True)
    )


# ------------------------------------------------------------
# EXECUTION  <- edit search parameters here
# ------------------------------------------------------------
config = STACSearchConfig(
    date_start      = "2024-06-01",
    date_end        = "2024-09-30",
    max_cloud_cover = 10,
    max_results     = 20,
)

searcher  = STACSceneSearch(config)
scenes_df = searcher.search(aoi_info)
summarise_results(scenes_df)

deduplicator = STACSceneDeduplicator(scenes_df)
clean_df     = deduplicator.deduplicate()
summarise_deduplicated(scenes_df, clean_df)


In [ ]:
# 3. Band download via stackstac COG (Cloud Optimised GeoTIFF)
import stackstac
import numpy as np
import rasterio
from rasterio.crs import CRS
from affine import Affine
import geopandas as gpd
from pathlib import Path
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import pandas as pd
from typing import Optional
import warnings
warnings.filterwarnings("ignore")


@dataclass
class STACBandDownloadConfig:
    """
    Configuration for COG band downloads via stackstac.

    Attributes
    ----------
    output_dir  : Root directory for raw band GeoTIFFs.
    resolution  : Output pixel size in metres. All bands are resampled to this
                  resolution by stackstac (20 m matches the native B8A resolution).
    n_threads   : Number of parallel download threads.
    bands       : Tuple of band identifiers to download.
    epsg        : EPSG code of the output CRS (32630 = UTM zone 30N).
    """
    output_dir:  Path
    resolution:  int   = 20
    n_threads:   int   = 4
    bands: tuple = (
        "B02",   # Blue        -- used in EVI
        "B03",   # Green       -- used in NDWI
        "B04",   # Red         -- used in NDVI, SAVI, EVI
        "B05",   # Red Edge 1 (705 nm) -- used in NDRE
        "B8A",   # NIR narrow  -- used in all indices
        "B11",   # SWIR1
        "B12",   # SWIR2       -- used in NBR
    )
    epsg: int = 32630


class STACCOGBandDownloader:
    """
    Downloads Sentinel-2 L2A bands as float32 GeoTIFFs using stackstac COG reads.

    stackstac issues HTTP range requests against the COG files, transferring only
    the byte ranges that intersect the specified bounding box. This avoids
    downloading the full ~1 GB SAFE archive per scene.

    Output structure on disk:
        <output_dir>/<YYYY-MM-DD>/<MGRS_tile>/<band>.tif
    """

    def __init__(self, config: STACBandDownloadConfig) -> None:
        self._config     = config
        self._write_lock = Lock()
        self._config.output_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def _affine_from_dataarray(data, resolution: int) -> Affine:
        """
        Build an Affine transform from the DataArray x/y coordinate arrays.

        stackstac 0.5.1 removed the array_to_affine helper. Coordinates in a
        stackstac DataArray represent pixel centres, so the raster origin
        (upper-left corner) is offset by half a pixel from the extremes.
        """
        res      = resolution
        x_origin = float(data.x.min()) - res / 2
        y_origin = float(data.y.max()) + res / 2
        return Affine(res, 0.0, x_origin, 0.0, -res, y_origin)

    def _download_scene_stackstac(
        self,
        stac_item,
        scene_date:  str,
        tile_id:     str,
        bbox_latlon: list,
    ) -> dict:
        """
        Download all configured bands for a single MGRS tile and date.

        Parameters
        ----------
        stac_item   : Signed STAC Item carrying COG asset URLs.
        scene_date  : Acquisition date string (YYYY-MM-DD).
        tile_id     : MGRS tile identifier (e.g. T30SVF).
        bbox_latlon : Spatial filter as [W, S, E, N] in EPSG:4326.
        """
        output_scene_dir = self._config.output_dir / scene_date / tile_id
        output_scene_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n  Reading bands via COG for {tile_id} ({scene_date})...")

        try:
            # bounds_latlon expects EPSG:4326 coordinates.
            # Using `bounds` (without suffix) would require UTM and produce an empty result.
            # fill_value must match dtype exactly -- stackstac 0.5.1 enforces strict type checking.
            stack = stackstac.stack(
                items         = [stac_item],
                assets        = list(self._config.bands),
                resolution    = self._config.resolution,
                epsg          = self._config.epsg,
                bounds_latlon = bbox_latlon,
                dtype         = "float32",
                rescale       = False,
                fill_value    = np.float32(0),
            )
        except Exception as exc:
            raise RuntimeError(f"stackstac.stack failed for {tile_id}: {exc}")

        print(f"  Downloading {len(self._config.bands)} bands ({self._config.resolution} m)...")
        try:
            data = stack.compute()  # xarray DataArray [time, band, y, x]
        except Exception as exc:
            raise RuntimeError(f"COG compute failed for {tile_id}: {exc}")

        transform = self._affine_from_dataarray(data, self._config.resolution)
        crs       = CRS.from_epsg(self._config.epsg)

        downloaded = {}

        for band_name in self._config.bands:
            out_path = output_scene_dir / f"{band_name}.tif"

            if out_path.exists():
                print(f"  SKIP: {band_name} ({tile_id}) already exists.")
                downloaded[band_name] = out_path
                continue

            try:
                band_data = data.sel(band=band_name).values  # [time, y, x]
                arr       = band_data[0]                      # single acquisition

                profile = {
                    "driver"    : "GTiff",
                    "dtype"     : "float32",
                    "count"     : 1,
                    "height"    : arr.shape[0],
                    "width"     : arr.shape[1],
                    "crs"       : crs,
                    "transform" : transform,
                    "compress"  : "lzw",
                    "tiled"     : True,
                    "blockxsize": 256,
                    "blockysize": 256,
                    "nodata"    : 0.0,
                }

                with self._write_lock:
                    with rasterio.open(out_path, "w", **profile) as dst:
                        dst.write(arr, 1)

                downloaded[band_name] = out_path
                print(f"  OK: {band_name} ({tile_id}) -> {out_path}")

            except Exception as exc:
                print(f"  ERROR: {band_name} ({tile_id}) -- {exc}")

        return downloaded

    def download_all_scenes(
        self,
        clean_df: pd.DataFrame,
        aoi_info: dict,
    ) -> dict:
        """Iterate over all dates and tiles in clean_df and download every configured band."""
        if clean_df.empty:
            raise ValueError("clean_df is empty. Run Cell 2 first.")

        required_cols = {"stac_item", "scene_name", "date", "tile_id"}
        missing = required_cols - set(clean_df.columns)
        if missing:
            raise ValueError(f"clean_df is missing columns: {missing}")

        if "bbox_stac" not in aoi_info:
            raise ValueError("aoi_info is missing 'bbox_stac'. Run Cell 1 first.")

        bbox_latlon = aoi_info["bbox_stac"]
        dates       = clean_df["date"].unique()
        all_results = {}
        total       = len(dates)

        for d_idx, date in enumerate(sorted(dates)):
            date_scenes = clean_df[clean_df["date"] == date]
            tiles       = date_scenes["tile_id"].tolist()
            print(f"\n[{d_idx+1}/{total}] {date} -- {len(tiles)} tile(s): {tiles}")

            all_results[date] = {}

            for _, row in date_scenes.iterrows():
                tile_id   = row["tile_id"]
                stac_item = row["stac_item"]

                try:
                    result = self._download_scene_stackstac(
                        stac_item, date, tile_id, bbox_latlon
                    )
                    all_results[date][tile_id] = result
                except RuntimeError as exc:
                    print(f"  ERROR: {tile_id} skipped -- {exc}")
                    continue

        print("\n" + "=" * 60)
        print(f"Download complete. {len(all_results)} dates processed.")
        print(f"Output directory: {self._config.output_dir}")
        print("=" * 60)

        return all_results


# ------------------------------------------------------------
# EXECUTION
# ------------------------------------------------------------
download_config = STACBandDownloadConfig(
    output_dir = Path("./output/raw_bands"),
    resolution = 20,
    n_threads  = 4,
    epsg       = 32630,   # UTM zone 30N -- adjust for your region
)

downloader = STACCOGBandDownloader(download_config)
all_bands  = downloader.download_all_scenes(clean_df, aoi_info)


In [ ]:
# 4. Spectral index computation -- NDVI, SAVI, EVI, NBR, NDRE, NDWI
# Output: one clipped float32 LZW GeoTIFF per index per scene (multi-tile aware)
import numpy as np
import rasterio
from rasterio.mask import mask as rasterio_mask
from rasterio.merge import merge as rasterio_merge
from rasterio.io import MemoryFile
import geopandas as gpd
from pathlib import Path
from dataclasses import dataclass
from threading import Lock
from scipy.ndimage import zoom
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)


@dataclass
class IndexComputationConfig:
    """
    Configuration for spectral index computation.

    Attributes
    ----------
    bands_root   : Root directory containing raw band GeoTIFFs (Cell 3 output).
    output_root  : Destination directory for computed index GeoTIFFs.
    aoi_path     : Path to the AOI GeoPackage used for polygon clipping.
    savi_L       : Soil adjustment factor L for SAVI (0 = dense canopy, 1 = sparse).
    nodata_value : Fill value for pixels outside the AOI polygon and invalid pixels.
    n_threads    : Number of parallel threads for index writing.
    """
    bands_root:   Path
    output_root:  Path
    aoi_path:     Path
    savi_L:       float = 0.5
    nodata_value: float = -9999.0
    n_threads:    int   = 1


class SpectralIndexComputer:
    """
    Computes NDVI, SAVI, EVI, NBR, NDRE, and NDWI from Sentinel-2 L2A bands.

    Multi-tile aware: when an AOI spans more than one MGRS tile, bands from all
    tiles are merged with rasterio before clipping to the exact polygon boundary.
    Pixels outside the polygon are set to nodata_value (-9999).

    Index formulae (bands scaled to surface reflectance: DN / 10 000):
        NDVI  = (B8A - B04) / (B8A + B04)
        SAVI  = ((B8A - B04) / (B8A + B04 + L)) * (1 + L)
        EVI   = 2.5 * (B8A - B04) / (B8A + 6*B04 - 7.5*B02 + 1)
        NBR   = (B8A - B12) / (B8A + B12)
        NDRE  = (B8A - B05) / (B8A + B05)
        NDWI  = (B03 - B8A) / (B03 + B8A)
    """

    def __init__(self, config: IndexComputationConfig) -> None:
        self._config     = config
        self._write_lock = Lock()
        self._config.output_root.mkdir(parents=True, exist_ok=True)
        self._aoi_geom   = self._load_aoi_geometry()

    def _load_aoi_geometry(self) -> list:
        if not self._config.aoi_path.exists():
            raise FileNotFoundError(f"AOI GeoPackage not found: {self._config.aoi_path}")
        gdf = gpd.read_file(self._config.aoi_path)
        if gdf.empty or gdf.geometry.is_empty.all():
            raise ValueError("AOI GeoPackage contains no valid geometries.")
        if gdf.crs is None:
            raise ValueError("AOI GeoPackage has no CRS defined.")
        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
        return [geom.__geo_interface__ for geom in gdf.geometry]

    def _merge_and_clip_band(self, band_paths: list[Path]) -> tuple[np.ndarray, dict]:
        """
        Merge multiple tile GeoTIFFs and clip to the exact AOI polygon boundary.

        Pixels outside the polygon are filled with nodata_value. When the AOI
        spans a single tile, the merge step is skipped. The clip is performed
        in the band's native CRS after reprojecting the AOI geometries to match.
        """
        from shapely.geometry import shape

        for p in band_paths:
            if not p.exists():
                raise FileNotFoundError(f"Band file not found: {p}")

        if len(band_paths) == 1:
            with rasterio.open(band_paths[0]) as src:
                data    = src.read(1).astype(np.float32)
                profile = src.profile.copy()
                src_crs = src.crs
        else:
            datasets = [rasterio.open(p) for p in band_paths]
            try:
                merged, transform = rasterio_merge(datasets)
                profile = datasets[0].profile.copy()
                profile.update(
                    height    = merged.shape[1],
                    width     = merged.shape[2],
                    transform = transform,
                )
                src_crs = datasets[0].crs
                data    = merged[0].astype(np.float32)
            finally:
                for ds in datasets:
                    ds.close()

        mem_profile = profile.copy()
        mem_profile.update(dtype="float32", driver="GTiff")

        with MemoryFile() as memfile:
            with memfile.open(**mem_profile) as mem_dst:
                mem_dst.write(data, 1)

            with memfile.open() as mem_src:
                aoi_shapes = [shape(g) for g in self._aoi_geom]
                aoi_gdf    = gpd.GeoDataFrame(geometry=aoi_shapes, crs="EPSG:4326")
                if src_crs.to_epsg() != 4326:
                    aoi_gdf = aoi_gdf.to_crs(src_crs)

                geoms = [g.__geo_interface__ for g in aoi_gdf.geometry]

                try:
                    clipped, clip_transform = rasterio_mask(
                        mem_src, geoms, crop=True,
                        nodata      = self._config.nodata_value,
                        filled      = True,
                        all_touched = True,
                    )
                except Exception as exc:
                    raise RuntimeError(f"rasterio.mask failed: {exc}")

                out_profile = mem_src.profile.copy()
                out_profile.update(
                    driver     = "GTiff",
                    dtype      = "float32",
                    count      = 1,
                    nodata     = self._config.nodata_value,
                    height     = clipped.shape[1],
                    width      = clipped.shape[2],
                    transform  = clip_transform,
                    compress   = "lzw",
                    tiled      = True,
                    blockxsize = 256,
                    blockysize = 256,
                )

                return clipped[0].astype(np.float32), out_profile

    def _resample_to_target(self, arr: np.ndarray, target_shape: tuple) -> np.ndarray:
        if arr.shape == target_shape:
            return arr.astype(np.float32)
        zoom_y = target_shape[0] / arr.shape[0]
        zoom_x = target_shape[1] / arr.shape[1]
        return zoom(arr, (zoom_y, zoom_x), order=1).astype(np.float32)

    def _mask_nodata(self, *arrays: np.ndarray) -> np.ndarray:
        """
        Boolean mask -- True where a pixel is invalid and must be excluded.

        A pixel is marked invalid if:
          - DN <= 0  : covers Sentinel-2 native nodata (DN = 0) and pixels outside
                       the AOI polygon filled by rasterio_mask with -9999.
                       Valid L2A surface reflectance always has DN > 0.
          - ~isfinite: NaN or Inf from arithmetic edge cases.
        """
        invalid = np.zeros(arrays[0].shape, dtype=bool)
        for arr in arrays:
            invalid |= (arr <= 0) | ~np.isfinite(arr)
        return invalid

    def _compute_indices(
        self,
        b02: np.ndarray, b03: np.ndarray, b04: np.ndarray,
        b05: np.ndarray, b8a: np.ndarray, b12: np.ndarray,
        nodata_mask: np.ndarray,
    ) -> dict[str, np.ndarray]:
        scale = 10000.0
        blue  = b02 / scale
        green = b03 / scale
        red   = b04 / scale
        re1   = b05 / scale
        nir   = b8a / scale
        swir  = b12 / scale
        nd    = self._config.nodata_value

        with np.errstate(divide="ignore", invalid="ignore"):
            denom = nir + red
            ndvi  = np.where(denom != 0, (nir - red) / denom, nd).astype(np.float32)

            L     = self._config.savi_L
            denom = nir + red + L
            savi  = np.where(denom != 0, ((nir - red) / denom) * (1.0 + L), nd).astype(np.float32)

            denom = nir + 6.0 * red - 7.5 * blue + 1.0
            evi   = np.where(denom != 0, 2.5 * (nir - red) / denom, nd).astype(np.float32)

            denom = nir + swir
            nbr   = np.where(denom != 0, (nir - swir) / denom, nd).astype(np.float32)

            denom = nir + re1
            ndre  = np.where(denom != 0, (nir - re1) / denom, nd).astype(np.float32)

            denom = green + nir
            ndwi  = np.where(denom != 0, (green - nir) / denom, nd).astype(np.float32)

        for arr in [ndvi, savi, evi, nbr, ndre, ndwi]:
            arr[nodata_mask] = nd

        def clamp(arr, lo, hi):
            return np.where((arr != nd) & ((arr < lo) | (arr > hi)), nd, arr).astype(np.float32)

        return {
            "NDVI": clamp(ndvi, -1.0, 1.0),
            "SAVI": clamp(savi, -1.5, 1.5),
            "EVI" : clamp(evi,  -1.0, 1.0),
            "NBR" : clamp(nbr,  -1.0, 1.0),
            "NDRE": clamp(ndre, -1.0, 1.0),
            "NDWI": clamp(ndwi, -1.0, 1.0),
        }

    def _write_index(
        self, index_name: str, array: np.ndarray,
        profile: dict, output_path: Path
    ) -> None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with self._write_lock:
            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(array, 1)
        print(f"  OK: {index_name} -> {output_path}")

    def process_date(self, date: str, tile_ids: list[str]) -> dict[str, Path]:
        print(f"\nProcessing date: {date} -- tiles: {tile_ids}")

        bands_config = [
            ("B8A", "20 m reference"),
            ("B02", "10 m Blue"),
            ("B03", "10 m Green"),
            ("B04", "10 m Red"),
            ("B05", "20 m Red Edge 1"),
            ("B12", "20 m SWIR2"),
        ]

        band_arrays = {}
        ref_profile = None

        for band, desc in bands_config:
            band_paths = [
                self._config.bands_root / date / tile / f"{band}.tif"
                for tile in tile_ids
            ]
            print(f"  Merging {band} ({len(band_paths)} tile(s))...")
            arr, profile = self._merge_and_clip_band(band_paths)
            band_arrays[band] = arr
            if band == "B8A":
                ref_profile = profile

        target_shape = band_arrays["B8A"].shape
        b02_arr = self._resample_to_target(band_arrays["B02"], target_shape)
        b03_arr = self._resample_to_target(band_arrays["B03"], target_shape)
        b04_arr = self._resample_to_target(band_arrays["B04"], target_shape)
        b05_arr = band_arrays["B05"].astype(np.float32)
        b8a_arr = band_arrays["B8A"].astype(np.float32)
        b12_arr = band_arrays["B12"].astype(np.float32)

        nodata_mask = self._mask_nodata(b02_arr, b03_arr, b04_arr, b05_arr, b8a_arr, b12_arr)
        indices     = self._compute_indices(b02_arr, b03_arr, b04_arr, b05_arr, b8a_arr, b12_arr, nodata_mask)

        output_dir   = self._config.output_root / date
        output_paths = {}

        for index_name, index_arr in indices.items():
            output_path = output_dir / f"{index_name}.tif"
            if output_path.exists():
                print(f"  SKIP: {index_name} already exists.")
                output_paths[index_name] = output_path
                continue
            self._write_index(index_name, index_arr, ref_profile, output_path)
            output_paths[index_name] = output_path

        return output_paths

    def process_all_scenes(self, clean_df: pd.DataFrame) -> dict:
        if clean_df.empty:
            raise ValueError("clean_df is empty. Run Cell 2 first.")

        all_results = {}
        dates = sorted(clean_df["date"].unique())

        for d_idx, date in enumerate(dates):
            tile_ids = clean_df[clean_df["date"] == date]["tile_id"].tolist()
            print(f"\n[{d_idx+1}/{len(dates)}] {date} -- {len(tile_ids)} tile(s): {tile_ids}")

            try:
                result = self.process_date(date, tile_ids)
                all_results[date] = result
            except (FileNotFoundError, ValueError, RuntimeError) as exc:
                print(f"  ERROR: {exc} -- skipping date.")
                continue

        print("\n" + "=" * 60)
        print("Index computation complete.")
        print(f"Dates processed : {len(all_results)}")
        print(f"Output directory: {self._config.output_root}")
        print("=" * 60)

        return all_results


def plot_scene_indices(indices_root: Path) -> None:
    """
    Generate a spatial map + histogram dashboard for every computed scene.

    For each date directory under indices_root the function reads the six index
    GeoTIFFs and produces a two-row figure: spatial maps on top, pixel-value
    histograms with descriptive statistics below. The figure is saved as a PNG
    alongside the GeoTIFFs.
    """
    index_cfg = {
        "NDVI": {"vmin": -0.1, "vmax": 0.8, "cmap": "RdYlGn", "label": "NDVI"},
        "SAVI": {"vmin": -0.1, "vmax": 0.8, "cmap": "RdYlGn", "label": "SAVI"},
        "EVI" : {"vmin": -0.1, "vmax": 0.8, "cmap": "RdYlGn", "label": "EVI"},
        "NBR" : {"vmin": -0.5, "vmax": 0.8, "cmap": "RdYlGn", "label": "NBR"},
        "NDRE": {"vmin": -0.1, "vmax": 0.7, "cmap": "RdYlGn", "label": "NDRE"},
        "NDWI": {"vmin": -0.5, "vmax": 0.5, "cmap": "RdBu",   "label": "NDWI"},
    }

    scene_dirs = sorted([
        d for d in indices_root.rglob("*")
        if d.is_dir() and any(d.glob("NDVI.tif"))
    ])

    if not scene_dirs:
        print(f"No scenes found in {indices_root}")
        return

    for scene_dir in scene_dirs:
        scene_date = scene_dir.name  # directory name is the acquisition date

        index_data = {}
        for index_name in ["NDVI", "SAVI", "EVI", "NBR", "NDRE", "NDWI"]:
            tif_path = scene_dir / f"{index_name}.tif"
            if not tif_path.exists():
                continue
            with rasterio.open(tif_path) as src:
                arr        = src.read(1).astype(np.float32)
                nodata     = src.nodata
                valid_mask = (arr != nodata) & np.isfinite(arr)
                index_data[index_name] = {
                    "arr"       : arr,
                    "valid_mask": valid_mask,
                    "valid"     : arr[valid_mask],
                    "transform" : src.transform,
                    "crs"       : src.crs,
                }

        n_indices = len(index_data)
        if n_indices == 0:
            continue

        fig = plt.figure(figsize=(5 * n_indices, 8))
        fig.suptitle(
            f"Spectral Index Dashboard -- {scene_date}",
            fontsize=11, fontweight="bold", y=1.01
        )

        gs = gridspec.GridSpec(2, n_indices, figure=fig, hspace=0.45, wspace=0.35)

        for col, (index_name, data) in enumerate(index_data.items()):
            cfg   = index_cfg[index_name]
            arr   = data["arr"]
            valid = data["valid"]
            mask  = data["valid_mask"]

            ax_map  = fig.add_subplot(gs[0, col])
            display = np.where(mask, arr, np.nan)
            im      = ax_map.imshow(
                display, cmap=cfg["cmap"],
                vmin=cfg["vmin"], vmax=cfg["vmax"], aspect="auto"
            )
            plt.colorbar(im, ax=ax_map, fraction=0.046, pad=0.04)
            ax_map.set_title(cfg["label"], fontsize=11, fontweight="bold")
            ax_map.set_xticks([]); ax_map.set_yticks([])

            ax_hist = fig.add_subplot(gs[1, col])
            if len(valid) > 0:
                ax_hist.hist(
                    valid, bins=40,
                    color="#2ecc71", edgecolor="#27ae60", alpha=0.85, linewidth=0.5
                )
                ax_hist.axvline(
                    valid.mean(), color="#e74c3c", lw=1.5, linestyle="--",
                    label=f"Mean {valid.mean():.3f}"
                )
                ax_hist.axvline(
                    np.median(valid), color="#3498db", lw=1.5, linestyle=":",
                    label=f"Median {np.median(valid):.3f}"
                )
                stats_text = (
                    f"Min    : {valid.min():.3f}\n"
                    f"Max    : {valid.max():.3f}\n"
                    f"Mean   : {valid.mean():.3f}\n"
                    f"Std    : {valid.std():.3f}\n"
                    f"Pixels : {len(valid)}"
                )
                ax_hist.text(
                    0.02, 0.97, stats_text,
                    transform=ax_hist.transAxes, fontsize=7,
                    verticalalignment="top",
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
                )
            ax_hist.set_xlabel(cfg["label"], fontsize=9)
            ax_hist.set_ylabel("Pixel count", fontsize=9)
            ax_hist.legend(fontsize=7, loc="upper right")
            ax_hist.tick_params(labelsize=8)

        plt.tight_layout()
        out_png = indices_root / f"dashboard_{scene_date}.png"
        plt.savefig(out_png, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Dashboard saved: {out_png}")


# ------------------------------------------------------------
# EXECUTION -- edit configuration below
# ------------------------------------------------------------
index_config = IndexComputationConfig(
    bands_root  = Path("./output/raw_bands"),
    output_root = Path("./output/indices"),
    aoi_path    = Path("path/to/your/aoi.gpkg"),  # <- same file as Cell 1
    savi_L      = 0.5,
    n_threads   = 4,
)

computer    = SpectralIndexComputer(index_config)
all_indices = computer.process_all_scenes(clean_df)

indices_root = Path("./output/indices")
plot_scene_indices(indices_root)
